# Лабораторная работа 10. Фильтрация и сглаживание сигналов

**Цель.** Выполнить частотную фильтрацию аудиосигнала и сравнить варианты экспоненциального сглаживания.


## Ход работы

В первой части используется фильтрация через спектр, во второй — сглаживание обычного временного ряда. Это два разных способа уменьшить влияние шума.


## Подготовка

Подключаю функции для чтения WAV-файла и быстрого преобразования Фурье. Также задаю варианты расположения аудиофайла, чтобы один и тот же ноутбук запускался в разных папках.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.fft import rfft, rfftfreq, irfft

np.random.seed(42)
audio_candidates = [Path('Wav_26mb.wav'), Path('data/Wav_26mb.wav')]
audio_path = next((path for path in audio_candidates if path.exists()), audio_candidates[0])


## Частотная фильтрация

Если аудиосигнал доступен, он переводится в спектр, после чего частоты выше 2000 Гц обнуляются. Так получается простая low-pass фильтрация.


In [ ]:
if not audio_path.exists():
    print('Аудиофайл не найден, блок фильтрации пропущен.')
else:
    samplerate, data = wavfile.read(audio_path)
    if data.ndim > 1:
        data = data[:, 0]
    time = np.arange(len(data)) / samplerate
    spectrum = rfft(data)
    freqs = rfftfreq(len(data), 1 / samplerate)
    filtered = spectrum.copy()
    filtered[freqs > 2000] = 0
    restored = irfft(filtered, n=len(data))

    print('Sampling Rate:', samplerate)
    print('Audio Shape:', data.shape)
    plt.figure(figsize=(12, 5))
    plt.plot(time[:5000], data[:5000], label='исходный сигнал', alpha=0.7)
    plt.plot(time[:5000], restored[:5000], label='после low-pass FFT', alpha=0.7)
    plt.legend()
    plt.xlabel('Время, с')
    plt.ylabel('Амплитуда')
    plt.show()


## Экспоненциальное сглаживание

На синтетическом ряде показываю, как `ewm` сглаживает локальный шум. В отличие от обычного среднего, новые точки получают больший вес, чем старые.


In [ ]:
# Демонстрация экспоненциального сглаживания на воспроизводимом синтетическом ряде.
t = np.arange(120)
series = pd.Series(0.05 * t + np.sin(t / 6) + np.random.normal(scale=0.25, size=len(t)))
smoothed = series.ewm(alpha=0.25, adjust=False).mean()

plt.figure(figsize=(10, 4))
plt.plot(series, label='исходный ряд')
plt.plot(smoothed, label='экспоненциальное сглаживание')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Вывод

Частотная фильтрация работает через спектр сигнала, а экспоненциальное сглаживание остаётся во временной области. Оба подхода уменьшают шум, но делают это по разным принципам.
